In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)
import sys
# sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
sys.path.append(r"/ihome/ylee/yiz133/Code/Data processing/functions/")
import mdata_utils 
import TCR_embedings

2026-05-12 17:18:11.925241: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-12 17:18:11.925592: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 17:18:11.963667: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-12 17:18:15.433061: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

In [3]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse

In [4]:
path = r"/ix1/ylee/Yifan_Zhang/Code_data/external/COMBAT paired T/"
filename = "common_HV200_atchEmb.h5mu"
DATA_PATH = path + filename

mdata_ori = mu.read(DATA_PATH)
mdata = mdata_ori.copy()

In [7]:
mdata

MuData object with n_obs × n_vars = 320152 × 2477
  obs:	'cloned', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length'
  uns:	'tcr_embs_feature_names'
  obsm:	'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_composition', 'tcr_embs'
  2 modalities
    gex:	320152 x 2477
      obs:	'Annotation_cluster_id', 'Annotation_cluster_name', 'Annotation_minor_subset', 'Annotation_major_subset', 'Annotation_cell_type', 'GEX_region', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'COMBAT_ID', 'scRNASeq_sample_ID', 'COMBAT_participant_timepoint_ID', 'Source', 'Age', 'Sex', 'Race', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'Respiratorysupport', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'Ethnicity', 'Tissue', 'DiseaseClassification', 'Pool_ID', 'Channel_ID', 'clone_id_size', 'clone_id'
      var:	'gene_ids', 'feature_types', 'hvg_union_COMBAT_ID'
      uns:	'Institute', 'ObjectCreateDate', 'Source_colors', 'Technology', 'genome_annotation_version', 'hvg_union_COMBAT_ID_k200', 'hvg_union_COMBAT_ID_meta'
      obsm:	'X_umap', 'X_umap_source'
      layers:	'raw'
    airr:	320152 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing'
      uns:	'chain_indices', 'scirpy_version'
      obsm:	'airr', 'chain_indices'

In [12]:
mdata['gex'].obs['DiseaseClassification'].value_counts()

DiseaseClassification
COVID-19;MONDO:0100096     218765
Sepsis;HP:0100806           53676
NA                          41449
Influenza;MONDO:0005812      6262
Name: count, dtype: int64

In [ ]:
# subseting on single pair
mdata = mdata[mdata['airr'].obs['chain_pairing'].isin(['single pair'])]
# mdata = mdata[mdata['gex'].obs['COMBAT_ID'].astype(int) > 20]
mdata = mdata_utils.sync_mdata_obs(mdata)


In [ ]:
## select one cell from each clonotype
mdata['airr'].obs['clone_id_size'] = mdata['gex'].obs['clone_id_size'] 
mdata['airr'].obs['clone_id'] = mdata['gex'].obs['clone_id'] 

airr_obs = mdata['airr'].obs.copy()
clone_id_col = airr_obs['clone_id']
if hasattr(clone_id_col, 'cat'):
    clone_id_col = clone_id_col.astype(str)
airr_obs['_clone_id_str'] = clone_id_col

expanded = airr_obs[airr_obs['clone_id_size'] > 1].dropna(subset=['_clone_id_str'])
sampled_expanded_idx = (
    expanded
    .groupby('_clone_id_str', observed=True)
    .sample(n=1, random_state=42)
    .index
)

single_idx = airr_obs[airr_obs['clone_id_size'] == 1].index
keep_idx = sampled_expanded_idx.append(single_idx)

mdata = mdata[keep_idx].copy()
print(f"Expanded clones: {len(expanded)} cells -> {len(sampled_expanded_idx)} (1 per clone)")
print(f"Single clones: {len(single_idx)}")
print(f"Total after dedup: {mdata.n_obs} cells")


In [ ]:
# Select the obs column used to split mdata into category-specific CCA runs.
# Change this to any column in mdata.obs or mdata['gex'].obs.
selected_obs = 'DiseaseClassification'
selected_obs_modality = 'gex'

train_frac = 0.8
dim_cca_max = 5
min_category_cells = 50
random_state = 42


def get_obs_series(mdata_in, obs_col, modality='gex'):
    if obs_col in mdata_in.obs.columns:
        return mdata_in.obs[obs_col]
    if modality in mdata_in.mod and obs_col in mdata_in[modality].obs.columns:
        return mdata_in[modality].obs[obs_col]
    raise KeyError(f"{obs_col!r} was not found in mdata.obs or mdata[{modality!r}].obs")


def as_dense_float_matrix(x):
    if issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=float)


def run_cca_for_subset(
    mdata_subset,
    category,
    train_frac=0.8,
    dim_cca_max=5,
    random_state=42,
):
    """Fit rCCA for one mdata subset and return arrays plus the annotated subset."""
    gex = mdata_subset['gex']
    view_gene_raw = as_dense_float_matrix(gex.X)
    view_tcr_raw = np.asarray(mdata_subset.obsm['tcr_embs'], dtype=float)

    rng = np.random.default_rng(random_state)
    train_mask = rng.random(mdata_subset.n_obs) < train_frac
    test_mask = ~train_mask

    if train_mask.sum() < 2 or test_mask.sum() < 2:
        raise ValueError(f"Category {category!r} does not have enough train/test cells")

    gene_scaler = StandardScaler()
    tcr_scaler = StandardScaler()
    view_gene_train = gene_scaler.fit_transform(view_gene_raw[train_mask])
    view_gene_test = gene_scaler.transform(view_gene_raw[test_mask])
    view_tcr_train = tcr_scaler.fit_transform(view_tcr_raw[train_mask])
    view_tcr_test = tcr_scaler.transform(view_tcr_raw[test_mask])

    dim_cca = min(view_gene_train.shape[1], view_tcr_train.shape[1], dim_cca_max)
    cca = TCR_embedings.rCCA(n_components=dim_cca, alpha_x=1, alpha_y=1)
    view_gene_c_train, view_tcr_c_train = cca.fit_transform(view_gene_train, view_tcr_train)
    view_gene_c_test, view_tcr_c_test = cca.transform(view_gene_test, view_tcr_test)

    gene_cv_scaler = StandardScaler()
    tcr_cv_scaler = StandardScaler()
    view_gene_c_train = gene_cv_scaler.fit_transform(view_gene_c_train)
    view_gene_c_test = gene_cv_scaler.transform(view_gene_c_test)
    view_tcr_c_train = tcr_cv_scaler.fit_transform(view_tcr_c_train)
    view_tcr_c_test = tcr_cv_scaler.transform(view_tcr_c_test)

    view_gene_c = np.zeros((mdata_subset.n_obs, dim_cca))
    view_tcr_c = np.zeros((mdata_subset.n_obs, dim_cca))
    view_gene_c[train_mask] = view_gene_c_train
    view_gene_c[test_mask] = view_gene_c_test
    view_tcr_c[train_mask] = view_tcr_c_train
    view_tcr_c[test_mask] = view_tcr_c_test

    mdata_subset = mdata_subset.copy()
    mdata_subset.obs['set'] = np.where(train_mask, 'train', 'test')
    cv_list = []
    for i in range(dim_cca):
        cv_name = f'CV_score_{i}'
        mdata_subset['gex'].obs[cv_name] = view_tcr_c[:, i]
        mdata_subset['gex'].obs[f'DCCA_CV_{i}'] = view_tcr_c[:, i]
        cv_list.append(cv_name)

    corr_train = np.array([
        np.corrcoef(view_gene_c_train[:, i], view_tcr_c_train[:, i])[0, 1]
        for i in range(dim_cca)
    ])
    corr_test = np.array([
        np.corrcoef(view_gene_c_test[:, i], view_tcr_c_test[:, i])[0, 1]
        for i in range(dim_cca)
    ])

    return {
        'category': category,
        'mdata': mdata_subset,
        'cca': cca,
        'dim_cca': dim_cca,
        'train_mask': train_mask,
        'test_mask': test_mask,
        'view_gene_c': view_gene_c,
        'view_tcr_c': view_tcr_c,
        'view_gene_c_train': view_gene_c_train,
        'view_tcr_c_train': view_tcr_c_train,
        'view_gene_c_test': view_gene_c_test,
        'view_tcr_c_test': view_tcr_c_test,
        'corr_train': corr_train,
        'corr_test': corr_test,
        'cv_list': cv_list,
    }


obs_values = get_obs_series(mdata, selected_obs, selected_obs_modality).astype(str)
category_counts = obs_values.value_counts(dropna=False)
selected_categories = category_counts[category_counts >= min_category_cells].index.tolist()

print(f"Selected obs: {selected_obs}")
print(f"Categories with >= {min_category_cells} cells: {len(selected_categories)}")
category_counts.loc[selected_categories]

## CCA

In [ ]:
cca_results = {}
cca_errors = {}

for category in selected_categories:
    category_mask = obs_values == category
    mdata_category = mdata[category_mask.to_numpy()].copy()

    try:
        result = run_cca_for_subset(
            mdata_category,
            category=category,
            train_frac=train_frac,
            dim_cca_max=dim_cca_max,
            random_state=random_state,
        )
        cca_results[category] = result
        print(
            f"{category}: n={mdata_category.n_obs}, "
            f"train={result['train_mask'].sum()}, test={result['test_mask'].sum()}, "
            f"dim_cca={result['dim_cca']}"
        )
    except Exception as exc:
        cca_errors[category] = exc
        print(f"Skipped {category}: {exc}")

cca_summary_df = pd.DataFrame([
    {
        'category': category,
        'n_cells': result['mdata'].n_obs,
        'n_train': int(result['train_mask'].sum()),
        'n_test': int(result['test_mask'].sum()),
        'dim_cca': result['dim_cca'],
        'mean_corr_train': float(np.nanmean(result['corr_train'])),
        'mean_corr_test': float(np.nanmean(result['corr_test'])),
    }
    for category, result in cca_results.items()
])

cca_summary_df

In [ ]:
### Deep CCA: fit ###
sys.path.append(r"/ihome/ylee/yiz133/Code/Tools/")

# from cca_zoo.deep import DCCA, architectures
# from cca_zoo.deep.data import NumpyDataset, check_dataset, get_dataloaders
# import lightning.pytorch as pl

# LATENT_DIMS = dim_cca
# EPOCHS = 30
# BATCH_SIZE = 64

# train_dataset = NumpyDataset([view_gene_train, view_tcr_train])
# test_dataset = NumpyDataset([view_gene_test, view_tcr_test])

# train_loader = get_dataloaders(train_dataset, batch_size=BATCH_SIZE)
# test_loader = get_dataloaders(test_dataset, batch_size=BATCH_SIZE)

# encoder_1 = architectures.Encoder(
#     latent_dimensions=LATENT_DIMS, feature_size=view_gene_train.shape[1],
# )
# encoder_2 = architectures.Encoder(
#     latent_dimensions=LATENT_DIMS, feature_size=view_tcr_train.shape[1],
# )

# dcca = DCCA(latent_dimensions=LATENT_DIMS, encoders=[encoder_1, encoder_2])
# trainer = pl.Trainer(
#     max_epochs=EPOCHS,
#     enable_checkpointing=True,
#     enable_model_summary=True,
#     enable_progress_bar=True,
# )
# trainer.fit(dcca, train_loader)

# from torch.utils.data import DataLoader
# train_loader_full = DataLoader(NumpyDataset([view_gene_train, view_tcr_train]), batch_size=BATCH_SIZE, drop_last=False, shuffle=False)
# test_loader_full = DataLoader(NumpyDataset([view_gene_test, view_tcr_test]), batch_size=BATCH_SIZE, drop_last=False, shuffle=False)
# view_gene_c_train, view_tcr_c_train = cca.transform(train_loader_full)
# view_gene_c_test, view_tcr_c_test = cca.transform(test_loader_full)

In [ ]:
# Pick one category result for the existing downstream plotting / modeling cells.
if not cca_results:
    raise RuntimeError('No category-specific CCA runs completed successfully.')

active_category = next(iter(cca_results))
active_result = cca_results[active_category]

mdata_cca = active_result['mdata']
cca = active_result['cca']
dim_cca = active_result['dim_cca']
train_mask = active_result['train_mask']
test_mask = active_result['test_mask']
view_gene_c = active_result['view_gene_c']
view_tcr_c = active_result['view_tcr_c']
view_gene_c_train = active_result['view_gene_c_train']
view_tcr_c_train = active_result['view_tcr_c_train']
view_gene_c_test = active_result['view_gene_c_test']
view_tcr_c_test = active_result['view_tcr_c_test']
cv_list = active_result['cv_list']

print(f"Active CCA category: {active_category}")

In [ ]:
print("Per-component correlations for active category (train / test):")
for i in range(dim_cca):
    r_tr = active_result['corr_train'][i]
    r_te = active_result['corr_test'][i]
    print(f"  CC {i}: train {r_tr:.4f}  test {r_te:.4f}")

print(f"\nTrain canonical variates shape: {view_gene_c_train.shape}")
print(f"Test canonical variates shape:  {view_gene_c_test.shape}")

In [ ]:
cv_list

In [ ]:
df_cca_weights = pd.DataFrame(cca.y_weights_)
cca_weights_abs = df_cca_weights.abs()

# Optional alpha/beta summary for the active category, using the existing embedding layout.
cca_weights_abs_beta = cca_weights_abs.iloc[0:596].sum() + cca_weights_abs.iloc[1191:1128].sum()
cca_weights_abs_alpha = cca_weights_abs.iloc[596:596+596].sum() + cca_weights_abs.iloc[1128:1374].sum()
df_cca_weights_abs_sum = pd.DataFrame(
    [cca_weights_abs_beta, cca_weights_abs_alpha],
    index=['beta', 'alpha']
)

df_cca_weights_abs_sum = df_cca_weights_abs_sum / df_cca_weights_abs_sum.sum(axis=0)
df_cca_weights_abs_sum

In [ ]:
fig, axes = plt.subplots(1, dim_cca, figsize=(6 * dim_cca, 6))
if dim_cca == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.scatter(
        view_gene_c_test[:, i], view_tcr_c_test[:, i],
        alpha=0.4, s=8,
    )
    ax.set_xlabel(f'Gene canonical variate {i} (test)')
    ax.set_ylabel(f'TCR canonical variate {i} (test)')
    ax.set_title(f'{active_category} test set (CC {i})')

plt.tight_layout()
plt.show()

In [ ]:
corr_train, corr_test = TCR_embedings.train_test_corr(view_gene_c_train, view_tcr_c_train, view_gene_c_test, view_tcr_c_test)

In [ ]:
ax = TCR_embedings.plot_train_test_corr(corr_train, corr_test)
# ax.set_ylim(0, 1)

## logit regression

In [ ]:
# Placeholder cell kept intentionally empty.

In [ ]:
### Logistic Regression per subtype: predict clone_status using DCCA canonical variates ###
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import torch
import torch.nn as nn

tg = target_groups if 'target_groups' in globals() else [
    'Tumor_multiSite_clone',
    'Tumor_singleSite_clone',
]
g1, g2 = tg[0], tg[1]

mdata_model = mdata_cca if 'mdata_cca' in globals() else mdata
cs = mdata_model.obs['clone_status'].astype(str).to_numpy()
sub = mdata_model.obs['subtype'].astype(str).to_numpy()
in_groups = np.isin(cs, tg)

y_all = (cs == g1).astype(int)

subtypes = sorted(set(sub[in_groups]))

class WeightedLogistic(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.feature_weights = nn.Parameter(torch.ones(n_features))
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x * self.feature_weights).squeeze(-1)

lr_results = []

for st in subtypes:
    st_mask = sub == st
    mask_tr = train_mask & in_groups & st_mask
    mask_te = test_mask & in_groups & st_mask

    X_tr = view_tcr_c[mask_tr]
    X_te = view_tcr_c[mask_te]
    y_tr = y_all[mask_tr]
    y_te = y_all[mask_te]

    if X_tr.shape[0] < 10 or X_te.shape[0] < 4:
        continue
    if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
        continue

    ros = RandomOverSampler(random_state=42)
    X_tr_bal, y_tr_bal = ros.fit_resample(X_tr, y_tr)

    # Trainable-weight logistic
    wlr_epochs = 200
    X_tr_t = torch.tensor(X_tr_bal, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr_bal, dtype=torch.float32)
    X_te_t = torch.tensor(X_te, dtype=torch.float32)

    wlr = WeightedLogistic(X_tr_t.shape[1])
    optimizer = torch.optim.Adam(wlr.parameters(), lr=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()

    wlr.train()
    for epoch in range(wlr_epochs):
        optimizer.zero_grad()
        loss_fn(wlr(X_tr_t), y_tr_t).backward()
        optimizer.step()

    wlr.eval()
    with torch.no_grad():
        y_pred_proba = torch.sigmoid(wlr(X_te_t)).numpy()
        y_pred = (y_pred_proba >= 0.5).astype(int)

    acc = accuracy_score(y_te, y_pred)
    try:
        auc = roc_auc_score(y_te, y_pred_proba)
    except ValueError:
        auc = np.nan

    learned_w = wlr.feature_weights.detach().numpy()

    lr_results.append({
        'subtype': st,
        'n_train': X_tr_bal.shape[0],
        'n_test': X_te.shape[0],
        'accuracy': acc,
        'roc_auc': auc,
        'feature_weights': learned_w.tolist(),
    })

    print(f"\n{'='*60}")
    print(f"Subtype: {st}  (train={X_tr_bal.shape[0]}, test={X_te.shape[0]})")
    print(f"{'='*60}")
    print(f"Feature weights: {learned_w}")
    print(f"Accuracy: {acc:.4f}   ROC AUC: {auc:.4f}")
    print(classification_report(y_te, y_pred, target_names=[g2, g1], zero_division=0))

lr_results_df = pd.DataFrame(lr_results)
lr_results_df
